# Colab — Fine-tune YOLOv8s (pipeline DL, H6)

Entrena YOLOv8s sobre las **20 clases del subset D2S** usando los splits filtrados que ya están en Drive (`train.json` / `val.json`).

**Requiere GPU**. *Runtime → Change runtime type → GPU* antes de Run All.

## Resume robusto

`data/processed/` se monta como **symlink directo a Drive**. Ultralytics escribe sus runs (`yolo_runs/yolov8s_d2s20/`) directamente en Drive. Tras un corte de Colab:
- Los checkpoints `last.pt` y `best.pt` ya están en Drive.
- Reabres el notebook, Run All, ultralytics levanta el `last.pt` y continúa (siempre que pases `--weights .../last.pt` al script o reanudes manualmente).

## Prerequisitos en Drive

En `MyDrive/grocery-detection/`:

```
raw/
    d2s_images_v*.tar.xz
    d2s_annotations_v*.tar.xz
processed/
    train.json                ← output de prepare_splits.py
    val.json
    test.json
```

## 1. Verificar GPU

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(out.stdout if out.returncode == 0 else "⚠️ No GPU. Cambia el runtime a GPU antes de continuar.")

## 2. Bootstrap — clona repo + helpers

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/arturmoret/GroceryTracker.git"
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

sys.path.insert(0, REPO_DIR + "/scripts")
sys.path.insert(0, REPO_DIR + "/src")
os.chdir(REPO_DIR)

from colab_helper import (
    mount_drive, install_deps, setup_dataset,
    link_processed_to_drive, run_script,
)
print("helpers cargados")

## 3. Montar Drive + instalar deps (incluido ultralytics)

In [ ]:
mount_drive()
install_deps()  # opencv-contrib + sklearn + pyyaml + rich

# Extra deps para H6: ultralytics trae torch + sus deps.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics"],
    check=True,
)
print("ultralytics instalado")

In [ ]:
setup_dataset()  # extrae D2S a /content/d2s + symlink data/d2s

In [ ]:
link_processed_to_drive()  # data/processed/ ↔ Drive — yolo_runs/ y yolo_best.pt caen directo allí

## 4. Preparar dataset YOLO

Genera `data/d2s/labels/*.txt` (junto a las imágenes) + `data/processed/yolo/dataset.yaml`. Idempotente — se puede re-correr cada sesión sin problema.

In [ ]:
rc = run_script("scripts/prepare_yolo_dataset.py")
assert rc == 0

## 5. Entrenar

Hyperparams en `configs/deep_yolo.yaml`. Para 20 clases + ~3000 imgs train, YOLOv8s ronda **2-3 h** en T4 free para 100 épocas (con early stopping suele cortar antes).

### Reanudar después de un corte

Si Colab muere a mitad de entreno, descomenta la segunda celda más abajo apuntando a `last.pt` que quedó en Drive:

In [ ]:
rc = run_script("scripts/run_deep_train.py")
assert rc == 0

In [ ]:
# Para reanudar tras corte (manual):
# LAST = "data/processed/yolo_runs/yolov8s_d2s20/weights/last.pt"
# rc = run_script("scripts/run_deep_train.py", "--weights", LAST)
# assert rc == 0

## Listo

Al terminar, `data/processed/yolo_best.pt` (copia canónica del mejor checkpoint) y todo el run de ultralytics están en Drive.

Siguiente: `notebooks/colab_yolo_infer.ipynb` → predicciones sobre test → JSON COCO listo para evaluación común.